# 00. Contexto de Negócio, Alinhamento de Dores e Limitações da Base

## 1. O Cenário Fictício: Reunião de Kickoff (Ata de Alinhamento)

**Participantes:**
- **Mariana:** Gerente de Produto & Conteúdo
- **Carlos:** Diretor de Marketing
- **Analista de Dados:** (Você)

### Registro da Discussão & Decisões de Escopo:
- **Demanda Inicial de Mariana:** Mapear a distribuição do catálogo por tipo/gênero e identificar os gêneros com maior *retenção* para direcionar aquisições.
- **Intervenção Técnica do Analista:** Esclarecido que o dataset (`netflix_titles.csv`) contém exclusivamente **metadados de oferta do catálogo** (título, país, data de adição, gêneros, duração, etc.), não possuindo métricas de consumo, audiência ou *churn/retenção*.
- **Acordo de Reenquadramento:** 
  1. A medição de *retenção real* fica formalmente fora do escopo e marcada para uma fase futura (com dados internos da plataforma).
  2. O escopo atual focará na **evolução da oferta e diversidade de catálogo** (tipos, gêneros, duração e janelamento de lançamentos).

- **Demanda de Carlos:** Avaliar a expansão regional (produção fora dos EUA) e identificar padrões sazonais de adição para suporte a campanhas de marketing.
- **Direcionamento do Analista:** A evolução temporal considerará a data de entrada na plataforma (`date_added`). As análises regionais e de gêneros utilizarão a metodologia de **aparição por item-título** para tratar coproduções e múltiplos gêneros.

---

## 2. Perguntas de Negócio & Matriz de Hipóteses Definidas

| ID | Pergunta de Negócio | Hipótese Testável | Métrica Operacional | Variáveis Chave |
|---|---|---|---|---|
| **H1** | Como o mix de mídias evoluiu? | A proporção de Filmes vs. Séries adicionados sofria alteração estrutural ao longo dos anos? | % de Filmes vs. Séries por ano de `date_added` | `type`, `date_added` |
| **H2** | O catálogo reflete a expansão internacional? | O percentual de aparições de produções com participação fora dos EUA cresceu nas adições recentes? | % de aparições fora dos EUA por ano de `date_added` | `country`, `date_added` |
| **H3** | Qual a distribuição demográfica do catálogo? | Qual é a distribuição por classificação indicativa e como ela evoluiu ao longo do tempo? | Distribuição % de `rating` ao longo dos anos | `rating`, `date_added` |
| **H4** | Existe sazonalidade no abastecimento? | Há picos de adição em meses específicos do ano que indiquem sazonalidade operacional/marketing? | Volumetria total de adições agregada por mês (1-12) | `date_added` (mês) |
| **H5** | Como evoluiu a diversidade de gêneros? | A diversidade de gêneros mudou e a concentração nos principais gêneros reduziu com o tempo? | 1. Nº de gêneros únicos por ano.<br>2. % de participação acumulada dos Top 5 gêneros. | `listed_in`, `type`, `date_added` |
| **H6** | Houve mudança no formato/duração das mídias? | A duração média dos filmes e a quantidade de temporadas das séries variaram ao longo dos anos? | Média de duração em minutos (Filmes) e mediana de temporadas (Séries) por ano | `duration`, `type`, `date_added` |
| **H7** | A Netflix acelerou a aquisição de lançamentos recentes? | O tempo de defasagem (*lag*) entre o ano de produção e a adição na plataforma diminuiu nos últimos anos? | Diferença em anos: `date_added.year - release_year` | `release_year`, `date_added` |

---

## 3. Decisões Metodológicas & Governança de Dados

### 3.1. Tratamento de Variáveis Multivaloradas (`country` e `listed_in`)
- **Escolha Metodológica:** Aplicação do pipeline de *split/explode* nas colunas `country` e `listed_in`.
- **Trade-off e Impacto Estatístico:** Esta decisão altera a unidade de análise de *Título Único* para *Aparição País-Título* e *Aparição Gênero-Título*. 
  - Exemplo: Um filme associado a 3 gêneros gera 3 registros de análise.
- **Consequência:** A soma total das aparições por país e gênero ultrapassará o volume de títulos únicos da base original. Os percentuais de representatividade serão calculados estritamente sobre o **total de aparições**, e não sobre o total de linhas brutas.

### 3.2. Escopo de Tratamento de Dados no Pipeline ETL
- **Colunas no Escopo Ativo de Limpeza:** `type`, `date_added`, `release_year`, `rating`, `duration`, `country`, `listed_in`.
- **Coluna Excluída da Limpeza Nesta Fase (`cast` e `director`):** A coluna `cast` contêm listas extensas de atores e, por não estar associada a nenhuma das hipóteses (H1 a H7), **não passará por tratamento de desnormalização/explode nesta fase**, otimizando o pipeline.

### 3.3. Limitações Conhecidas da Base
1. **Fotografia Estática:** Trata-se de uma base estática extraída do Kaggle. A data de corte exata do catálogo será apurada dinamicamente no notebook `01_data_audit.ipynb` através do registro máximo em `date_added`.
2. **Ausência de Métricas de Consumo:** Base estritamente descritiva do inventário; não inclui métricas de audiência, engajamento ou receita.

---

## 4. Escopo Técnico do Projeto

* **Dentro do Escopo:** Pipeline ETL em Python, ingestão em banco PostgreSQL via SQLAlchemy, Auditoria de Qualidade de Dados, Análise Exploratória (EDA) descritiva/diagnóstica no VS Code e Storytelling de Negócio.
* **Fora do Escopo:** Algoritmos de Recomendação, Previsão de Churn, Análise de Sentimento (NLP), Modelagem Financeira.